# Focused AAD search and comparison (077/076)

Run this cell on Colab A100. It updates the checkout, uses pinned dependencies, then runs 16 architectures × two frame sizes × two FPS values (64 jobs), selects five finalists and trains ten comparison models from scratch. Validation freezes every checkpoint before test. No manual paths or toggles.

Search: 256-epoch cap; comparison: 512-epoch cap. Runtime is measured, not guaranteed. Check downloaded compact results ZIPs after each search block. Checkpoints stay on the VM unless separately exported; metrics-only ZIPs cannot restore full training/test evidence. Do not delete the VM before saving what you need.


In [ ]:
from pathlib import Path
import importlib
import os
import subprocess
import sys


def run_focused_workflow():
    """Run 64 balanced search jobs, then ten fresh comparisons and frozen tests."""
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    repo = Path("/content/phd_research")
    repo.parent.mkdir(parents=True, exist_ok=True)
    if not (repo / ".git").is_dir():
        if repo.exists() and any(repo.iterdir()):
            raise RuntimeError(
                f"{repo} is not an empty folder or Git checkout; existing files preserved."
            )
        subprocess.run(
            [
                "git",
                "clone",
                "https://github.com/sanelehlabisa/phd_research.git",
                str(repo),
            ],
            check=True,
        )
    else:
        update = subprocess.run(
            ["git", "pull", "--ff-only", "origin", "master"],
            cwd=repo,
            capture_output=True,
            text=True,
        )
        print(update.stdout, end="", flush=True)
        if update.returncode:
            raise RuntimeError(
                "Could not safely update checkout; resolve this message without discarding evidence:\n"
                + update.stderr
            )
    root = repo / "papers/001-journal-abnormal-activity-recognition/implementation"
    required = [
        "src/focused_protocol.py",
        "src/capacity_search.py",
        "src/study_matrix.py",
        "notebooks/utils/aad_study.py",
        "notebooks/utils/study_archive.py",
        "notebooks/utils/colab_bootstrap.py",
        "configs/experiments/aad_focused_search_colab.json",
        "configs/experiments/aad_shortlist_comparison_colab.json",
    ]
    missing = [p for p in required if not (root / p).is_file()]
    if missing:
        raise RuntimeError(
            f"Checkout is missing focused workflow files: {missing}. Publish the complete verified update before running this export."
        )
    revision = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        cwd=repo,
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
    print(
        f"Code checkout: {repo} @ {revision}\nNotebook Python: {sys.executable}",
        flush=True,
    )
    bootstrap = subprocess.run(
        [
            sys.executable,
            str(root / "notebooks/utils/colab_bootstrap.py"),
            str(root / "requirements.txt"),
        ]
    )
    if bootstrap.returncode == 75:
        print(
            "Pinned packages were repaired. Restart the Colab runtime once, reconnect the GPU, then rerun this cell.",
            flush=True,
        )
        return None
    if bootstrap.returncode:
        raise RuntimeError("Colab setup failed; see the bootstrap output above.")
    sys.path.insert(0, str(root))
    for name in list(sys.modules):
        if (
            name == "src"
            or name.startswith("src.")
            or name == "notebooks"
            or name.startswith("notebooks.")
        ):
            del sys.modules[name]
    importlib.invalidate_caches()
    from notebooks.utils.helpers import validate_runtime
    from notebooks.utils.aad_study import run_focused_aad_study

    print(validate_runtime(require_cuda=True), flush=True)
    print(
        "64 search + 10 comparison trainings; compact results downloads during execution. Keep checkpoints separately; browser delivery is unverified.",
        flush=True,
    )
    return run_focused_aad_study(root)


artifact_zip = run_focused_workflow()